# RoadShield Vision — Kaggle free-GPU training (RDD2022, real data)

Use Kaggle **Notebooks** → enable an available GPU accelerator (Settings). This is a reproducible real-data notebook, not a claim that a real training run has happened.
- The free GPU allocation/availability varies by account. Never purchase compute or run paid resources without approval.
- Enable Internet for the initial GitHub/Figshare access if permitted by your account. No secret tokens are required by this notebook.
- Start with one country (e.g. India) via official Figshare metadata and MD5 verification. Do not download the 9.9 GB Norway archive unless you need it.
- Persist `/kaggle/working/roadshield-run/` (including `last.pt`) as a **Kaggle output/dataset version** before ending the session; attach it to a new session in `/kaggle/input` to resume. Kaggle ephemeral working files do **not** survive by default.
- Never claim measured accuracy from a synthetic smoke test, and never automatically deploy a candidate model.
- Until PR #1 is merged, clone the specific feature branch with the training code.

In [ ]:
import torch
assert torch.cuda.is_available(), "No free Kaggle GPU attached. Enable a GPU accelerator or stop; don't train on CPU."
!nvidia-smi -L
!git clone --depth 1 --single-branch --branch claude/roadshield-ai-production-u7mtac https://github.com/Alaa2134/roadshield-ai.git
%cd roadshield-ai
# Keep Kaggle's CUDA torch; avoid blindly reinstalling it.
!pip -q install onnx onnxscript
from pathlib import Path
import shutil
OUT = Path('/kaggle/working/roadshield-run')
OUT.mkdir(parents=True, exist_ok=True)
# Optional: first add the saved output folder as a Kaggle input dataset/version.
PREVIOUS = Path('/kaggle/input/roadshield-checkpoints/roadshield-run')
if PREVIOUS.is_dir() and (PREVIOUS/'last.pt').is_file() and not (OUT/'last.pt').exists():
    for name in ('last.pt','model.pt','history.json'):
        if (PREVIOUS/name).is_file():
            shutil.copy2(PREVIOUS/name, OUT/name)
    print('Restored saved epoch checkpoint.')
print('Output directory:', OUT)

## 1. Download (official source, licence-checked)

In [ ]:
!python ml/datasets/download_rdd2022.py --out /kaggle/working/rdd2022 --dry-run
import json
record = json.load(open('/kaggle/working/rdd2022/LICENSE_RECORD.json'))
assert record['allowed'], "Dataset licence needs review."
print("Official Figshare files:", [(x['name'], round((x['size'] or 0)/1e6)) for x in record['files']])

In [ ]:
# Start with one country archive, when present in the official Figshare record.
# Check the printed file list first; this match is not an invented URL or mirror.
FILE_MATCH = "India"
assert any(FILE_MATCH.lower() in f['name'].lower() for f in record['files']), "Choose a match shown in the official Figshare file list"
!python ml/datasets/download_rdd2022.py --out /kaggle/working/rdd2022 --match {FILE_MATCH}
import zipfile
from pathlib import Path
DATA = Path('/kaggle/working/rdd2022')
archives = list(DATA.glob('*.zip'))
assert archives, "No matching official ZIP downloaded; check FILE_MATCH."
for archive in archives:
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DATA)
xml_dirs = [p for p in DATA.rglob('xmls') if p.is_dir() and p.parent.name == 'annotations' and p.parent.parent.name == 'train']
assert xml_dirs, "Cannot locate official VOC train/annotations/xmls structure; inspect ZIP layout."
RAW = str(xml_dirs[0].parents[3])
print("Detected dataset root:", RAW)

## 2. Prepare

In [ ]:
!python ml/datasets/prepare_rdd2022.py --raw {RAW} --out /kaggle/working/rdd2022_prepared --seed 42
import json
manifest = json.load(open('/kaggle/working/rdd2022_prepared/MANIFEST.json'))
print("Version:", manifest['version'], "Split counts:", manifest['counts'])
assert all(manifest['counts'].get(split, 0) > 0 for split in ('train','val','test')), "Cannot train on empty splits"

## 3–4. Train and evaluate

In [ ]:
import subprocess, sys
# Same config every resume (24 total epochs); conservative batch for a free T4.
cmd = [sys.executable, 'ml/detector/train_detector.py', '--data', '/kaggle/working/rdd2022_prepared',
       '--raw', RAW, '--out', str(OUT), '--epochs', '24', '--batch', '2', '--seed', '42', '--pretrained', '--onnx']
if (OUT / 'last.pt').exists():
    cmd.append('--resume')
print("Resuming existing training" if '--resume' in cmd else "Starting real RDD2022 training")
subprocess.run(cmd, check=True)

In [ ]:
import json
metrics_path = OUT / 'metrics.json'
assert metrics_path.is_file(), "No held-out metrics yet; training has not finished."
m = json.loads(metrics_path.read_text())
assert not m.get('smoke_test'), "This is smoke-test data, NOT a real quality metric."
print('Dataset version:', m['dataset_version'], 'Train/val/test:', m['split_sizes'])
print(json.dumps(m['test'], indent=2))
print('By country:', json.dumps(m['per_country'], indent=2))
print('Latency:', m['latency_ms'])

## Saving and resuming the real model

After a successful epoch, `OUT/last.pt` contains the model, optimiser, scheduler, RNG state and dataset configuration. `OUT/model.pt` is the best validation checkpoint. Save the **complete output directory** as a private Kaggle notebook output or input dataset so it can be reattached at the next free GPU session.

On a later notebook, attach that output as a dataset and set `PREVIOUS` to the actual mounted folder. The start cell restores it and the train cell adds `--resume` automatically. If interrupted mid-epoch, training restarts from the last completed epoch.

Raw RDD2022 data may need re-downloading if not persisted; you can also attach a private archive to `/kaggle/input`, but never bypass the official licence/MD5 check. Record actual GPU run and measured held-out results in `TRAINING_REPORT.md` before calling it trained.

In [ ]:
print('Training output:', OUT)
print([(p.name, round(p.stat().st_size/1e6, 1)) for p in sorted(OUT.iterdir()) if p.is_file()])
print('Save this output in Kaggle before the runtime ends. Model is a candidate only until reviewed.')